In [2]:
import sys
sys.path.append('../cmldask')
import os, pickle, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from statsmodels.stats.multitest import multipletests
from cmldask import CMLDask as da
from dask.distributed import wait, as_completed
import gc
import itertools

from ptsa.data.filters import ButterworthFilter, MorletWaveletFilter
import cmlreaders as cml
from compute_scalp_features import create_baseline_events

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option('display.max_columns', None)

SyntaxError: unterminated string literal (detected at line 14) (3464874176.py, line 14)

In [2]:
EXP = 'CourierReinstate1'
# SUBJECTS = ['LTP564', 'LTP565']
SUBJECTS = ['LTP564', 'LTP565', 'LTP566', 'LTP567', 'LTP568', 'LTP569', 'LTP571', 'LTP572', 'LTP573',
            'LTP574', 'LTP575', 'LTP576', 'LTP577', 'LTP578', 'LTP579', 'LTP580', 'LTP581', 'LTP583',
            'LTP584', 'LTP585', 'LTP586', 'LTP587', 'LTP588', 'LTP589', 'LTP590', 'LTP591', 'LTP592', 
            'LTP593', 'LTP594', 'LTP595', 'LTP596', 'LTP597', 'LTP598', 'LTP599', 'LTP600', 'LTP601', 
            'LTP602', 'LTP603', 'LTP604', 'LTP605']

RETR_REL_START = -900
RETR_REL_STOP  =  -50
DELIB_REL_START = -900
DELIB_REL_STOP  = -50
BUFFER_MS_DELIB = 833.333333333

WIDTH = 6 
FREQS = np.logspace(np.log10(2), np.log10(100), 46)
NOTCH_BAND = (58., 62.)
BATCH_EVENTS = 64
ROI_ORDER = ['LAI','RAI','LAS','RAS','LPS','RPS','LPI','RPI']

In [3]:
def assign_roi(channel):
    roi_dict = {
        'LAS':['C24','C25','D2','D3','D4','D11','D12','D13'],
        'LAI':['C31','C32','D5','D6','D9','D10','D21','D22'],
        'LPS':['D29','A5','A6','A7','A8','A17','A18'],
        'LPI':['D30','D31','A9','A10','A11','A15','A16'],
        'RAS':['B30','B31','B32','C2','C3','C4','C11','C12'],
        'RAI':['B24','B25','B28','B29','C5','C6','C9','C10'],
        'RPS':['A30','A31','A32','B3','B4','B5','B13'],
        'RPI':['A28','A29','B6','B7','B8','B11','B12'],
    }
    for roi, chans in roi_dict.items():
        if channel in chans:
            return roi
    return None

In [4]:
df_idx = cml.get_data_index('ltp').query("experiment == @EXP").copy()
if SUBJECTS is not None:
    df_idx = df_idx[df_idx['subject'].isin(SUBJECTS)]
subjects = sorted(df_idx.subject.unique())
print(f'Found {len(subjects)} subjects for experiment "{EXP}".')

Found 40 subjects for experiment "CourierReinstate1".


In [5]:
def welch_t_from_agg(n1, sum1, sumsq1, n0, sum0, sumsq0):
    """
    All arrays are shape (n_channels, n_freqs) or scalars broadcastable thereto.
    Returns t of shape (n_channels, n_freqs).
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        m1 = sum1 / np.maximum(n1, 1)
        m0 = sum0 / np.maximum(n0, 1)
        v1 = (sumsq1 - (sum1**2)/np.maximum(n1, 1)) / np.maximum(n1 - 1, 1)
        v0 = (sumsq0 - (sum0**2)/np.maximum(n0, 1)) / np.maximum(n0 - 1, 1)
        denom = np.sqrt(v1 / np.maximum(n1, 1) + v0 / np.maximum(n0, 1))
        t = (m1 - m0) / denom
        t[~np.isfinite(t)] = np.nan
    return t

In [6]:
def ensure_trial_col(df):
    if 'trial' in df.columns:
        return df
    for alt in ['list', 'trialno', 'trial_num', 'trial_number']:
        if alt in df.columns:
            return df.rename(columns={alt: 'trial'})
    # Fallback: make a synthetic trial within session (should rarely be needed)
    g = df.groupby(['subject','session'], sort=False).cumcount()
    out = df.copy()
    out['trial'] = g
    return out

In [7]:
REQUIRED_NUMERIC = ['session', 'trial', 'mstime']
OPTIONAL_NUMERIC = ['intrusion', 'serialpos', 'repetition', 'itemno', 'list', 'eegoffset']
STRING_COLS      = ['type', 'eegfile', 'phase']

def prep_for_baseline(evs: pd.DataFrame) -> pd.DataFrame:
    e = evs.copy()

    # Ensure we have a 'trial' column with a sensible name
    if 'trial' not in e.columns:
        for alt in ['list','trialno','trial_num','trial_number']:
            if alt in e.columns:
                e = e.rename(columns={alt: 'trial'})
                break

    e = e.replace(r'^\s*$', np.nan, regex=True)
    
    # Coerce numeric columns; empty strings -> NaN
    for col in REQUIRED_NUMERIC + OPTIONAL_NUMERIC:
        if col in e.columns:
            e[col] = pd.to_numeric(e[col], errors='coerce')

    # If present, intrusion NaN -> 0 (non-intrusions)
    if 'intrusion' in e.columns:
        e['intrusion'] = e['intrusion'].fillna(0)

    # Ensure anchors exist and drop rows missing anchors
    if not set(REQUIRED_NUMERIC).issubset(e.columns):
        return e.iloc[0:0].copy()
    e = e.dropna(subset=REQUIRED_NUMERIC)
    
    # Cast anchors to int64
    for col in REQUIRED_NUMERIC:
        e[col] = e[col].astype(np.int64)

    # Ensure required string columns exist and are string dtype
    for col in STRING_COLS:
        if col not in e.columns:
            e[col] = ''
        else:
            e[col] = e[col].fillna('').astype(str)

    return e

def df_to_recarray_with_dtype(df: pd.DataFrame) -> np.recarray:
    """
    Build a structured recarray with explicit dtypes:
      - REQUIRED_NUMERIC & OPTIONALLY numeric columns: int64/float64
      - STRING_COLS ('type','eegfile','phase'): Unicode strings
      - others: use a conservative string fallback
    """
    names, formats, arrays = [], [], []

    for name in df.columns:
        if name in STRING_COLS:
            arr = df[name].fillna('').astype(str).to_numpy()
            fmt = 'U128'
        else:
            dt = df[name].dtype
            if np.issubdtype(dt, np.integer):
                arr = df[name].astype(np.int64).to_numpy()
                fmt = np.int64
            elif np.issubdtype(dt, np.floating):
                arr = df[name].astype(np.float64).to_numpy()
                fmt = np.float64
            else:
                # Fallback to string for anything non-numeric
                arr = df[name].fillna('').astype(str).to_numpy()
                fmt = 'U128'
        names.append(name); formats.append(fmt); arrays.append(arr)

    # Create recarray without FutureWarning (avoid fromrecords with list-of-lists)
    rec = np.rec.fromarrays(arrays, names=names, formats=formats)
    return rec

In [1]:
def process_subject_stats(subject, EXP=EXP, batch=BATCH_EVENTS):
    """
    Returns DataFrame: subject, session, trial, type, outputpos, serialpos, item, freq, power
    One row = one REC_WORD (retrieval) or REC_BASE (matched deliberation) event at one freq
    Power averaged across channels
    """
    sessions = cml.get_data_index('ltp').query("experiment == @EXP and subject == @subject").session.unique()

    dfs = []
    freq_vec = np.array(FREQS)

    for sess in sessions:
        try:
            reader = cml.CMLReader(subject=subject, experiment=EXP, session=sess)
            evs_all = reader.load('task_events')
            evs_all = evs_all[(evs_all['type'].isin(['REC_WORD', 'REC_BASE'])) & (evs_all['eegoffset']>=0)]
            if evs_all.empty:
                continue
            evs_all = evs_all.sort_values(['trial', 'mstime']).reset_index(drop=True)
            
            evs_all['outputpos'] = np.nan
            rec_word_mask = evs_all['type' == 'REC_WORD']
            evs_all.loc[rec_word_mask, 'outputpos'] = evs_all.loc[rec_word_mask].groupby('trial').cumcount() + 1

            for start in range(0, len(evs_all), batch):
                evs = evs_all.iloc[start:start+batch]
                eeg = reader.load_eeg(evs, rel_start=-BUFFER_MS, rel_stop=REL_STOP+BUFFER_MS, clean='LCF').to_ptsa()

                # Notch filter
                eeg = ButterworthFilter(freq_range=list(NOTCH_BAND), filt_type='stop', order=4).filter(eeg)

                # Wavelet power
                mw = MorletWaveletFilter(freqs=freq_vec, width=WIDTH, output='power', complete=True)
                pwr = mw.filter(eeg).sel(time=slice(REL_START, REL_STOP)).mean(dim='time')
                p = pwr.transpose('event', 'channel', 'frequency').values
                p_event_freq = p.mean(axis=1)  # averaged across channels
                
                # Event-level metadata
                meta_cols = ['subject', 'session', 'trial', 'type', 'outputpos', 'serialpos', 'item']
                meta = evs[meta_cols]
                
                power_df = pd.DataFrame(p_event_freq, columns=freq_vec, index=meta.index)
                batch_df = pd.concat([meta.reset_index(drop=True), power_df.reset_index(drop=True)], axis=1)
                batch_long = batch_df.melt(id_vars=meta_cols, var_name='freq', value_name='power')
                dfs.append(batch_long)

                del eeg, pwr, p, p_event_freq, power_df, batch_df, batch_long
                gc.collect()

        except Exception as e:
            print(f"[WARN] {subject} session {sess} skipped: {e}")
            continue

    if not dfs:
        return pd.DataFrame(columns=['subject', 'session', 'trial', 'type', 'outputpos', 'serialpos', 'item', 'freq', 'power'])
    
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values(['subject', 'session', 'trial', 'type', 'outputpos', 'serialpos', 'freq']).reset_index(drop=True)
    
    return df

NameError: name 'EXP' is not defined